In [36]:
import pandas as pd 
import pennylane as qml 
import argparse
import pickle
import string
from typing import Iterable, Tuple

import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from tqdm import tqdm
import math 
import random
from torch.utils.data import Subset

In [37]:
ALPHABET = string.ascii_lowercase + string.digits + "."
char2idx = {c: i + 1 for i, c in enumerate(ALPHABET)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(char2idx) + 1

MAX_LEN = 50


class DomainDataset(Dataset):
    def __init__(self, samples: Iterable[Tuple[str, int]]):
        self.samples = list(samples)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        dom, lbl = self.samples[idx]
        x = domain_to_tensor(dom)
        return x, lbl
def domain_to_tensor(domain: str) -> torch.Tensor:
    arr = [char2idx.get(c, 0) for c in domain.lower()][:MAX_LEN]
    arr += [0] * (MAX_LEN - len(arr))
    return torch.tensor(arr, dtype=torch.long)


def tensor_to_domain(tensor: torch.Tensor) -> str:
    domain = "".join(idx2char.get(idx, "") for idx in tensor.tolist() if idx > 0)
    return domain

In [39]:
def vqc_block(n_qubits: int):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def circuit(inputs, weights):
        
        qml.AngleEmbedding(inputs, wires=range(n_qubits))
        for wire in range(n_qubits):
            qml.RY(weights[0, wire], wires=wire)
            qml.RZ(weights[1, wire], wires=wire)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])
        return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

    weight_shapes = {"weights": (2, n_qubits)}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


class QuantumGate(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_qubits: int):
        super().__init__()
        self.input_linear = nn.Linear(input_dim + hidden_dim, n_qubits)
        self.vqc = vqc_block(n_qubits)
        self.output_linear = nn.Linear(n_qubits, hidden_dim)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        combined = torch.cat([x_t, h_prev], dim=1)

        q_inputs = self.input_linear(combined)
        q_inputs = math.pi * torch.tanh(q_inputs) # limit the angle 
        q_features = self.vqc(q_inputs)
        return self.output_linear(q_features)


class QuantumLSTMCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_qubits: int):
        super().__init__()
        self.hidden_dim = hidden_dim        
        self.forget_gate = QuantumGate(input_dim, hidden_dim, n_qubits)
        self.input_gate = QuantumGate(input_dim, hidden_dim, n_qubits)
        self.candidate_gate = QuantumGate(input_dim, hidden_dim, n_qubits)
        self.output_gate = QuantumGate(input_dim, hidden_dim, n_qubits)

    def forward(self, x_t: torch.Tensor, state: Tuple[torch.Tensor, torch.Tensor]):
        h_prev, c_prev = state
        f_t = torch.sigmoid(self.forget_gate(x_t, h_prev))
        i_t = torch.sigmoid(self.input_gate(x_t, h_prev))
        g_t = torch.tanh(self.candidate_gate(x_t, h_prev))
        c_t = f_t * c_prev + i_t * g_t
        o_t = torch.sigmoid(self.output_gate(x_t, h_prev))
        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t


class QuantumLSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, n_qubits: int, num_classes: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.cell = QuantumLSTMCell(embed_dim, hidden_dim, n_qubits)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        batch_size,T ,E = emb.shape 
        h_t = emb.new_zeros((batch_size, self.cell.hidden_dim))
        c_t = emb.new_zeros((batch_size, self.cell.hidden_dim))
        hs =[]
        for t in range(emb.size(1)):
            h_t, c_t = self.cell(emb[:, t, :], (h_t, c_t))
            hs.append(h_t.unsqueeze(1))
        H_seq = torch.cat(hs,dim=1) 
        mask = (x!=0).float().unsqueeze(-1)   
        pooled = (H_seq*mask ).sum(dim= 1 )/(mask.sum(dim =1 )+1e-9 )
        logits = self.fc ( self.dropout (pooled ))
        
        return logits 

In [40]:
def inspect_vqc_flow(model: nn.Module,
                     domains,
                     t: int = 0,
                     gate: str = "candidate",
                     device: torch.device = torch.device("cpu"),
                     max_print: int = 1):
    """
    Quan sát dữ liệu chảy qua VQC của một gate trong QuantumLSTMCell.
    - model: QuantumLSTMClassifier đã khởi tạo/đưa lên device
    - domains: list[str] các domain thô để dựng batch
    - t: bước thời gian (ký tự thứ t trong chuỗi) để lấy x_t
    - gate: 'forget' | 'input' | 'candidate' | 'output'
    - device: cpu/cuda
    - max_print: số mẫu đầu tiên in minh họa

    Returns:
        dict với các khóa:
        - lin_out: trước khi tanh*π (B, n_qubits)
        - angles: sau tanh*π, đưa vào qml.AngleEmbedding (B, n_qubits)
        - vqc_out: expval PauliZ từ VQC (B, n_qubits)
        - post_lin: sau output_linear của QuantumGate (B, hidden_dim)
        - logits: logits cuối cùng của model (B, num_classes)
        - probs: softmax của logits (B, num_classes)
    """
    model.eval()
    cell = model.cell
    gate_name = {"forget": "forget_gate",
                 "input": "input_gate",
                 "candidate": "candidate_gate",
                 "output": "output_gate"}.get(gate, None)
    if gate_name is None:
        raise ValueError("gate phải là 'forget' | 'input' | 'candidate' | 'output'.")

    gate_mod = getattr(cell, gate_name)

    # Chuẩn bị batch từ domains
    x = torch.stack([domain_to_tensor(d) for d in domains], dim=0).to(device)
    with torch.no_grad():
        emb = model.embedding(x)  # (B, T, E)

    B, T, E = emb.shape
    if t < 0 or t >= T:
        raise ValueError(f"t ngoài phạm vi [0, {T-1}]")

    h_prev = torch.zeros((B, cell.hidden_dim), device=device, dtype=emb.dtype)
    c_prev = torch.zeros((B, cell.hidden_dim), device=device, dtype=emb.dtype)

    caches = {}

    def hook_input_linear(_mod, _inp, out):
        # out shape: (B, n_qubits)
        caches["lin_out"] = out.detach().clone()

    def hook_vqc(_mod, _inp, out):
        # angles = π * tanh(lin_out)
        angles = math.pi * torch.tanh(caches["lin_out"])
        caches["angles"] = angles.detach().clone()
        # out shape: (B, n_qubits) expval Z
        caches["vqc_out"] = out.detach().clone()

    def hook_output_linear(_mod, _inp, out):
        # out shape: (B, hidden_dim)
        caches["post_lin"] = out.detach().clone()

    h1 = gate_mod.input_linear.register_forward_hook(hook_input_linear)
    h2 = gate_mod.vqc.register_forward_hook(hook_vqc)
    h3 = gate_mod.output_linear.register_forward_hook(hook_output_linear)

    try:
        with torch.no_grad():
            # Chạy đúng 1 bước qua gate cần soi để lấy trung gian
            x_t = emb[:, t, :]  # (B, E)
            _ = gate_mod(x_t.to(device), h_prev.to(device))

            # Chạy full model để lấy logits/probs cho cùng batch
            logits = model(x.to(device))
            probs = torch.softmax(logits, dim=1)

            caches["logits"] = logits.detach().clone()
            caches["probs"] = probs.detach().clone()
    finally:
        # Tháo hook cho sạch
        h1.remove(); h2.remove(); h3.remove()

    # In nhanh vài phần tử để bạn nhìn hình dung
    k = min(max_print, B)
    for i in range(k):
        print(f"--- Sample {i} ---")
        print(f"domain: {tensor_to_domain(x[i].cpu())}")
        print(f"lin_out[{i}][:5]: {caches['lin_out'][i, :5].cpu().numpy()}")
        print(f"angles[{i}][:5] (radians): {caches['angles'][i, :5].cpu().numpy()}")
        print(f"vqc_out[{i}][:5]: {caches['vqc_out'][i, :5].cpu().numpy()}")
        print(f"post_lin[{i}][:5]: {caches['post_lin'][i, :5].cpu().numpy()}")
        print(f"logits[{i}]: {caches['logits'][i].cpu().numpy()}")
        print(f"probs [{i}]: {caches['probs'][i].cpu().numpy()}")

    return {k: v.cpu() for k, v in caches.items()}


In [41]:
model = QuantumLSTMClassifier(
        vocab_size=vocab_size,
        embed_dim=args.embed_dim,
        hidden_dim=args.hidden_dim,
        n_qubits=args.n_qubits,
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)

IndentationError: unexpected indent (1434374547.py, line 7)

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

sample_domains = ["google.com", "aslkdj1.com", "xn--pypal-9ta.biz"]
dump = inspect_vqc_flow(model, sample_domains, t=0, gate="candidate", device=device, max_print=3)

# Bạn cũng có thể xem toàn bộ tensor:
dump["angles"].shape, dump["vqc_out"].shape, dump["post_lin"].shape


NameError: name 'model' is not defined

In [43]:
# ---------------------------------------------------------
# DEMO: Biến đổi dữ liệu qua từng bước của pipeline
# Dùng lại các hàm/lớp đã định nghĩa ở trên:
# - domain_to_tensor, tensor_to_domain
# - DomainDataset
# - QuantumLSTMClassifier
# ---------------------------------------------------------

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# 1) Dữ liệu thô (domain,label) như bạn đưa
raw_lines = [
    "ns3.p15.dynect.net,                                                   0",
    "1lkvihapejevelqlq6cnsb3.ddns.net,                        1",
    "easzuphjjv.us-east-1.awsapprunner.com,            0",
    "rrwpyxihfvmqsyhs.bid,                                             1",
    "a257.casalemedia.com,                                            0",
]

# 2) Parse thành (domain_str, label_int)
def parse_lines(lines):
    samples = []
    for ln in lines:
        # tách theo dấu phẩy, strip khoảng trắng
        parts = ln.split(",")
        domain = parts[0].strip()
        label = int(parts[1].strip())
        samples.append((domain, label))
    return samples

samples = parse_lines(raw_lines)

print("==> BƯỚC 1: Sau khi parse (domain, label)")
for i, (d, y) in enumerate(samples):
    print(f"{i:02d} | domain='{d}' | label={y}")
print()

# 3) Dùng DomainDataset để tự động domain_to_tensor
mini_ds = DomainDataset(samples)

print("==> BƯỚC 2: domain -> tensor chỉ số (len=50, 0 là PAD)")
for i in range(len(mini_ds)):
    x_i, y_i = mini_ds[i]  # x_i: LongTensor [50]
    # In vài thông tin: độ dài, số pad, vài phần tử đầu
    pad_count = int((x_i == 0).sum().item())
    nonzero = x_i[x_i != 0][:10].tolist()
    recon = tensor_to_domain(x_i)  # decode để check
    print(f"{i:02d} | label={y_i} | shape={tuple(x_i.shape)} | pad={pad_count} | first_nonzero={nonzero}")
    print(f"     decode_back: '{recon}'")
print()

# 4) Gom batch và tạo DataLoader để giống pipeline thực chiến
loader = DataLoader(mini_ds, batch_size=4, shuffle=False, drop_last=False)

# 5) Khởi tạo model (tham số nhẹ để chạy nhanh)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantumLSTMClassifier(
    vocab_size=vocab_size,
    embed_dim=16,      # bạn có thể đổi về args.embed_dim trong code chính
    hidden_dim=8,      # hoặc args.hidden_dim
    n_qubits=4,        # hoặc args.n_qubits
    num_classes=2
).to(device)
model.eval()

print("==> BƯỚC 3: Embedding -> QuantumLSTM -> Masked mean pooling -> Logits/Prob")
with torch.no_grad():
    for b_idx, (x_batch, y_batch) in enumerate(loader):
        x_batch = x_batch.to(device)  # [B, 50]
        y_batch = y_batch.to(device)

        # 3.1 Embedding
        emb = model.embedding(x_batch)      # [B, T=50, E]
        print(f"[Batch {b_idx}] embedding shape: {tuple(emb.shape)}  (B,T,E)")

        # 3.2 Chạy qua toàn bộ forward của model để có logits
        logits = model(x_batch)              # [B, 2]
        probs = F.softmax(logits, dim=1)     # [B, 2]

        # 3.3 Mask dùng trong model: (x!=0).float().unsqueeze(-1)
        mask = (x_batch != 0).float().unsqueeze(-1)  # [B,T,1]
        valid_steps = mask.sum(dim=1).squeeze(-1)    # [B,1] -> [B]
        print(f"[Batch {b_idx}] valid_steps (không tính PAD) per sample:", valid_steps.squeeze(-1).tolist())

        # 3.4 In kết quả
        pred = logits.argmax(dim=1)
        for i in range(x_batch.size(0)):
            idx_global = b_idx * loader.batch_size + i
            dom = samples[idx_global][0]
            print(f"   -> sample {idx_global:02d} | '{dom}' | label={int(y_batch[i].item())} | "
                  f"logits={logits[i].cpu().tolist()} | probs={probs[i].cpu().tolist()} | pred={int(pred[i].item())}")
        print()

print("Hoàn tất pipeline demo.")


==> BƯỚC 1: Sau khi parse (domain, label)
00 | domain='ns3.p15.dynect.net' | label=0
01 | domain='1lkvihapejevelqlq6cnsb3.ddns.net' | label=1
02 | domain='easzuphjjv.us-east-1.awsapprunner.com' | label=0
03 | domain='rrwpyxihfvmqsyhs.bid' | label=1
04 | domain='a257.casalemedia.com' | label=0

==> BƯỚC 2: domain -> tensor chỉ số (len=50, 0 là PAD)
00 | label=0 | shape=(50,) | pad=32 | first_nonzero=[14, 19, 30, 37, 16, 28, 32, 37, 4, 25]
     decode_back: 'ns3.p15.dynect.net'
01 | label=1 | shape=(50,) | pad=18 | first_nonzero=[28, 12, 11, 22, 9, 8, 1, 16, 5, 10]
     decode_back: '1lkvihapejevelqlq6cnsb3.ddns.net'
02 | label=0 | shape=(50,) | pad=15 | first_nonzero=[5, 1, 19, 26, 21, 16, 8, 10, 10, 22]
     decode_back: 'easzuphjjv.useast1.awsapprunner.com'
03 | label=1 | shape=(50,) | pad=30 | first_nonzero=[18, 18, 23, 16, 25, 24, 9, 8, 6, 22]
     decode_back: 'rrwpyxihfvmqsyhs.bid'
04 | label=0 | shape=(50,) | pad=30 | first_nonzero=[1, 29, 32, 34, 37, 3, 1, 19, 1, 12]
     decode